# SynthSplat — 3DGS Training (Colab)

Trains a 3D Gaussian Splatting model on synthetic images from the SynthSplat renderer.

**Setup**: Zip `renderer/output/`, upload to Google Drive, set runtime to T4 GPU, run all cells.

In [ ]:
#@title 1. Mount Google Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q gsplat torchmetrics tqdm matplotlib Pillow

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
#@title 2. Unzip Dataset
#@markdown Upload `output.zip` to Google Drive, then set the path below:
ZIP_PATH = "/content/drive/MyDrive/output.zip"  #@param {type:"string"}

import os, zipfile

DATA_DIR = "/content/data/output"
os.makedirs("/content/data", exist_ok=True)

if not os.path.exists(os.path.join(DATA_DIR, "cameras.json")):
    print(f"Extracting {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall("/content/data")
    # Handle case where zip contains output/ folder or files directly
    if not os.path.exists(os.path.join(DATA_DIR, "cameras.json")):
        # Maybe extracted to /content/data/ directly
        if os.path.exists("/content/data/cameras.json"):
            os.makedirs(DATA_DIR, exist_ok=True)
            !mv /content/data/cameras.json /content/data/frame_*.png {DATA_DIR}/
    print("Done.")
else:
    print("Dataset already extracted.")

import json
with open(os.path.join(DATA_DIR, "cameras.json")) as f:
    cams = json.load(f)
print(f"Found {len(cams)} camera views.")
print(f"Sample filenames: {[c['filename'] for c in cams[:3]]}")

In [ ]:
#@title 3. Dataset Loader
import json, math, os, torch, numpy as np
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as transforms


def _estimate_scene_bounds_fallback(cameras):
    pos = np.array([[c["position"][i] for i in range(3)] for c in cameras], dtype=np.float64)
    center = pos.mean(axis=0)
    dists = np.linalg.norm(pos - center, axis=1)
    orbit = float(np.median(dists))
    fov_deg = float(cameras[0].get("fov_degrees", 60.0))
    fov = math.radians(fov_deg)
    span_v = 2.0 * orbit * math.tan(fov * 0.5)
    radius = max(0.5, float(span_v * 0.45))
    return torch.tensor(center, dtype=torch.float32), radius


class SynthSplatDataset(Dataset):
    def __init__(self, output_dir, split='all'):
        self.output_dir = output_dir
        cameras_path = os.path.join(output_dir, 'cameras.json')
        if not os.path.exists(cameras_path):
            raise FileNotFoundError(f"cameras.json not found at {cameras_path}")
        with open(cameras_path, 'r') as f:
            self.cameras = json.load(f)
        fc = self.cameras[0]
        if "scene_center" in fc and "scene_radius" in fc:
            self.scene_center = torch.tensor(fc["scene_center"], dtype=torch.float32)
            self.scene_radius = float(fc["scene_radius"])
        else:
            self.scene_center, self.scene_radius = _estimate_scene_bounds_fallback(self.cameras)
        if split == 'test':
            self.cameras = self.cameras[::10]
        elif split == 'train':
            self.cameras = [c for i, c in enumerate(self.cameras) if i % 10 != 0]
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.cameras)

    def __getitem__(self, idx):
        cam = self.cameras[idx]
        H, W = cam['height'], cam['width']
        img = Image.open(os.path.join(self.output_dir, cam['filename'])).convert('RGB')
        if img.size != (W, H):
            _f = getattr(Image, "Resampling", Image).BILINEAR
            img = img.resize((W, H), _f)
        img = self.transform(img)
        view_matrix = torch.tensor(cam['view_matrix'], dtype=torch.float32)
        proj_matrix = torch.tensor(cam['projection_matrix'], dtype=torch.float32)
        c2w = torch.inverse(view_matrix)
        flip = torch.tensor([[1,0,0,0],[0,-1,0,0],[0,0,-1,0],[0,0,0,1]], dtype=torch.float32)
        c2w = c2w @ flip
        fx = float(torch.abs(proj_matrix[0, 0]) * (W / 2.0))
        fy = float(torch.abs(proj_matrix[1, 1]) * (H / 2.0))
        intrinsics = torch.tensor([fx, fy, W/2.0, H/2.0], dtype=torch.float32)
        return {'image': img, 'c2w': c2w, 'intrinsics': intrinsics,
                'height': H, 'width': W, 'frame_id': cam['frame_id']}


dataset = SynthSplatDataset(DATA_DIR, split='all')
print(f"Dataset: {len(dataset)} views, {dataset[0]['height']}x{dataset[0]['width']}")
print(f"Scene init: center={dataset.scene_center.numpy()}, radius={dataset.scene_radius:.4f}")

In [ ]:
#@title 4. Model & Training Config
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import numpy as np

from gsplat.rendering import rasterization

SH_DEGREE       = 3
NUM_SH_COEFFS   = (SH_DEGREE + 1) ** 2
INIT_NUM_POINTS = 30_000
NUM_ITERATIONS  = 30_000

DENSIFY_START    = 500
DENSIFY_END      = 25_000
DENSIFY_INTERVAL = 100
GRAD_THRESH      = 0.0001
MIN_OPACITY      = 0.002
MAX_SCALE_THRESH = 12.0
MAX_GAUSSIANS    = 800_000
RENDERER_BG_RGB  = (0.15, 0.35, 0.72)
RASTER_EPS2D     = 0.1
CLONE_SPLIT_SCALE = 0.008

LR_MEANS     = 1.6e-4
LR_SCALES    = 5e-3
LR_QUATS     = 1e-3
LR_OPACITIES = 5e-2
LR_SH_DC    = 2.5e-3
LR_SH_REST  = 1.25e-4


class GaussianModel:
    def __init__(self, num_points, device='cuda', init_center=None, init_radius=None):
        self.device = device
        pts = (torch.rand(num_points * 2, 3) - 0.5) * 2.0
        pts = pts[pts.norm(dim=1) < 1.0][:num_points]
        while pts.shape[0] < num_points:
            extra = (torch.rand(num_points, 3) - 0.5) * 2.0
            pts = torch.cat([pts, extra[extra.norm(dim=1) < 1.0]], 0)[:num_points]
        if init_center is not None and init_radius is not None:
            c = init_center.to(device).reshape(1, 3)
            pts = (c + float(init_radius) * pts.to(device)).requires_grad_(True)
        else:
            pts = pts.to(device).requires_grad_(True)
        self.means = pts
        self.scales = (torch.ones(num_points, 3, device=device) * -5.0).requires_grad_(True)
        self.quats = torch.rand(num_points, 4, device=device).requires_grad_(True)
        self.opacities = torch.zeros(num_points, 1, device=device).requires_grad_(True)
        sh = torch.zeros(num_points, NUM_SH_COEFFS, 3, device=device)
        sh[:, 0, :] = torch.rand(num_points, 3, device=device)
        self.sh_dc   = sh[:, :1, :].contiguous().clone().requires_grad_(True)
        self.sh_rest = sh[:, 1:, :].contiguous().clone().requires_grad_(True)
        self.means2d_grad_accum = torch.zeros(num_points, device=device)
        self.denom = torch.zeros(num_points, device=device)

    @property
    def num_gaussians(self): return self.means.shape[0]

    def sh_coeffs(self): return torch.cat([self.sh_dc, self.sh_rest], dim=1)

    @staticmethod
    def _replace_tensor_in_optimizer(optimizer, old_tensor, new_tensor):
        for group in optimizer.param_groups:
            for i, p in enumerate(group['params']):
                if p is old_tensor:
                    stored = optimizer.state.get(p, {})
                    del optimizer.state[p]
                    group['params'][i] = new_tensor
                    if stored:
                        for k, v in stored.items():
                            if isinstance(v, torch.Tensor) and v.shape == old_tensor.shape:
                                stored[k] = torch.zeros_like(new_tensor)
                        optimizer.state[new_tensor] = stored
                    return

    def _cat_tensors_to_optimizer(self, optimizer, ext_dict):
        for attr, ext in ext_dict.items():
            old = getattr(self, attr)
            new = torch.cat([old, ext], 0).requires_grad_(True)
            self._replace_tensor_in_optimizer(optimizer, old, new)
            setattr(self, attr, new)

    def _prune_tensors_in_optimizer(self, optimizer, mask):
        for attr in ['means','scales','quats','opacities','sh_dc','sh_rest']:
            old = getattr(self, attr)
            new = old[mask].clone().detach().requires_grad_(True)
            self._replace_tensor_in_optimizer(optimizer, old, new)
            setattr(self, attr, new)
        self.means2d_grad_accum = self.means2d_grad_accum[mask]
        self.denom = self.denom[mask]

    def densify_and_prune(self, optimizer, it):
        N = self.num_gaussians
        avg_grad = self.means2d_grad_accum / (self.denom + 1e-8)
        act_scales = torch.exp(self.scales)
        ms = act_scales.max(dim=1).values

        cm = (avg_grad >= GRAD_THRESH) & (ms <= CLONE_SPLIT_SCALE)
        if cm.sum() > 0 and (N + cm.sum()) <= MAX_GAUSSIANS:
            ext = {a: getattr(self, a)[cm].clone().detach() for a in
                   ['means','scales','quats','opacities','sh_dc','sh_rest']}
            self._cat_tensors_to_optimizer(optimizer, ext)
            nn_ = cm.sum().item()
            self.means2d_grad_accum = torch.cat([self.means2d_grad_accum, torch.zeros(nn_, device=self.device)])
            self.denom = torch.cat([self.denom, torch.zeros(nn_, device=self.device)])

        N = self.num_gaussians
        avg_grad = self.means2d_grad_accum / (self.denom + 1e-8)
        act_scales = torch.exp(self.scales)
        ms = act_scales.max(dim=1).values

        sm = (avg_grad >= GRAD_THRESH) & (ms > CLONE_SPLIT_SCALE)
        if sm.sum() > 0 and (N + sm.sum()) <= MAX_GAUSSIANS:
            stds = act_scales[sm]
            new_m = self.means[sm].detach().repeat(2,1) + torch.randn(sm.sum()*2, 3, device=self.device) * stds.repeat(2,1)
            ext = {'means': new_m,
                   'scales': (act_scales[sm]/1.6).log().repeat(2,1),
                   'quats': self.quats[sm].detach().repeat(2,1),
                   'opacities': self.opacities[sm].detach().repeat(2,1),
                   'sh_dc': self.sh_dc[sm].detach().repeat(2,1,1),
                   'sh_rest': self.sh_rest[sm].detach().repeat(2,1,1)}
            self._cat_tensors_to_optimizer(optimizer, ext)
            added = new_m.shape[0]
            self.means2d_grad_accum = torch.cat([self.means2d_grad_accum, torch.zeros(added, device=self.device)])
            self.denom = torch.cat([self.denom, torch.zeros(added, device=self.device)])
            keep = torch.ones(self.num_gaussians, dtype=torch.bool, device=self.device)
            keep[sm.nonzero(as_tuple=True)[0]] = False
            self._prune_tensors_in_optimizer(optimizer, keep)

        if it >= 3000:
            aop = torch.sigmoid(self.opacities.squeeze(-1))
            asc = torch.exp(self.scales).max(dim=1).values
            keep = (aop > MIN_OPACITY) & (asc < MAX_SCALE_THRESH)
            min_g = max(INIT_NUM_POINTS, 75_000)
            if keep.sum() < min_g:
                _, topk = aop.topk(min(min_g, self.num_gaussians))
                keep[topk] = True
            if (~keep).sum() > 0:
                self._prune_tensors_in_optimizer(optimizer, keep)

        if it % 3000 == 0 and it <= DENSIFY_END:
            new_op = torch.full_like(self.opacities, -2.0)
            old_op = self.opacities
            self._replace_tensor_in_optimizer(optimizer, old_op, new_op.requires_grad_(True))
            self.opacities = new_op.requires_grad_(True)

        self.means2d_grad_accum = torch.zeros(self.num_gaussians, device=self.device)
        self.denom = torch.zeros(self.num_gaussians, device=self.device)

    def state_dict(self):
        return {k: getattr(self, k).detach().cpu() for k in
                ['means','scales','quats','opacities','sh_dc','sh_rest']}

    def load_state_dict(self, sd):
        for k in ['means','scales','quats','opacities']:
            setattr(self, k, sd[k].to(self.device).requires_grad_(True))
        if 'sh_dc' in sd:
            self.sh_dc = sd['sh_dc'].to(self.device).requires_grad_(True)
            self.sh_rest = sd['sh_rest'].to(self.device).requires_grad_(True)
        else:
            sh = sd['sh_coeffs'].to(self.device)
            self.sh_dc = sh[:,:1,:].contiguous().requires_grad_(True)
            self.sh_rest = sh[:,1:,:].contiguous().requires_grad_(True)
        N = self.means.shape[0]
        self.means2d_grad_accum = torch.zeros(N, device=self.device)
        self.denom = torch.zeros(N, device=self.device)


def accumulate_means2d_grads_for_densify(model, info):
    if 'means2d' not in info:
        return
    m2d = info['means2d']
    g = getattr(m2d, 'absgrad', None)
    if g is None and m2d.grad is not None:
        g = m2d.grad
    if g is None:
        return
    g = g.clone().detach()
    W = float(info.get('width', 1))
    H = float(info.get('height', 1))
    nc = float(info.get('n_cameras', 1))
    g[..., 0] *= W / 2.0 * nc
    g[..., 1] *= H / 2.0 * nc
    gids = info.get('gaussian_ids')
    if gids is not None and gids.numel() > 0 and g.dim() <= 2:
        gn = g.norm(dim=-1)
        gids = gids.long()
        if int(gids.max().item()) >= model.num_gaussians:
            return
        ones = torch.ones_like(gn, dtype=model.denom.dtype)
        model.means2d_grad_accum.index_add_(0, gids, gn)
        model.denom.index_add_(0, gids, ones)
        return
    if g.dim() == 3:
        g = g[0]
    gn = g.norm(dim=-1)
    if gn.shape[0] == model.num_gaussians:
        model.means2d_grad_accum += gn
        model.denom += 1

print("Model & config loaded.")
print(f"  SH degree: {SH_DEGREE}  |  Iterations: {NUM_ITERATIONS}  |  Max Gaussians: {MAX_GAUSSIANS:,}")

In [ ]:
#@title 5. Train!
from torchmetrics.image import StructuralSimilarityIndexMeasure, PeakSignalNoiseRatio
import csv, time

device = torch.device('cuda')
CKPT_DIR = '/content/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

dataloader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=2, pin_memory=True)
model = GaussianModel(
    INIT_NUM_POINTS, device=device,
    init_center=dataset.scene_center, init_radius=dataset.scene_radius,
)

optimizer = optim.Adam([
    {'params': [model.means],     'lr': LR_MEANS,     'name': 'means'},
    {'params': [model.scales],    'lr': LR_SCALES,    'name': 'scales'},
    {'params': [model.quats],     'lr': LR_QUATS,     'name': 'quats'},
    {'params': [model.opacities], 'lr': LR_OPACITIES, 'name': 'opacities'},
    {'params': [model.sh_dc],     'lr': LR_SH_DC,     'name': 'sh_dc'},
    {'params': [model.sh_rest],   'lr': LR_SH_REST,   'name': 'sh_rest'},
], eps=1e-15)

ssim_fn = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
psnr_fn = PeakSignalNoiseRatio(data_range=1.0).to(device)

log_rows = []
it = 0
t0 = time.time()
pbar = tqdm(total=NUM_ITERATIONS, desc='Training')

while it < NUM_ITERATIONS:
    for batch in dataloader:
        if it >= NUM_ITERATIONS:
            break
        gt = batch['image'].to(device)
        c2w = batch['c2w'].to(device)
        intr = batch['intrinsics'].to(device)
        H, W = batch['height'].item(), batch['width'].item()

        viewmat = torch.inverse(c2w[0])
        fx, fy, cx, cy = intr[0]
        K = torch.zeros((3,3), device=device)
        K[0,0]=fx; K[0,2]=cx; K[1,1]=fy; K[1,2]=cy; K[2,2]=1.0

        a_s = torch.exp(model.scales)
        a_o = torch.sigmoid(model.opacities).squeeze(-1)
        a_q = model.quats / (model.quats.norm(dim=1, keepdim=True) + 1e-8)
        sh = model.sh_coeffs()
        bg = torch.tensor(RENDERER_BG_RGB, device=device, dtype=torch.float32)

        colors, alphas, info = rasterization(
            model.means, a_q, a_s, a_o, sh,
            viewmat.unsqueeze(0), K.unsqueeze(0),
            W, H,
            near_plane=0.01, far_plane=1e10, radius_clip=0.0,
            eps2d=RASTER_EPS2D, sh_degree=SH_DEGREE,
            absgrad=True, backgrounds=bg,
        )

        rendered = colors.permute(0,3,1,2)
        l1 = nn.functional.l1_loss(rendered, gt)
        ssim_l = 1.0 - ssim_fn(rendered, gt)
        loss = 0.9 * l1 + 0.1 * ssim_l

        optimizer.zero_grad()
        if 'means2d' in info:
            info['means2d'].retain_grad()
        loss.backward()
        accumulate_means2d_grads_for_densify(model, info)

        optimizer.step()
        it += 1
        pbar.update(1)
        pbar.set_description(f'Loss:{loss.item():.4f} N:{model.num_gaussians}')

        if DENSIFY_START <= it <= DENSIFY_END and it % DENSIFY_INTERVAL == 0:
            if it == DENSIFY_START:
                ag = model.means2d_grad_accum
                print(f'[densify debug] it={it} accum_max={ag.max():.6f} accum_nonzero={int((ag>0).sum())} denom_max={model.denom.max():.0f}')
            model.densify_and_prune(optimizer, it)
            if it == DENSIFY_START:
                print(f'[densify debug] after prune: num_gaussians={model.num_gaussians}')

        if it % 100 == 0:
            with torch.no_grad():
                p = psnr_fn(rendered, gt).item()
                s = ssim_fn(rendered, gt).item()
            log_rows.append((it, loss.item(), p, s, model.num_gaussians))

        if it % 5000 == 0:
            torch.save(model.state_dict(), f'{CKPT_DIR}/ckpt_{it}.pt')
            torch.cuda.empty_cache()

pbar.close()
elapsed = time.time() - t0
torch.save(model.state_dict(), f'{CKPT_DIR}/ckpt_final.pt')
print(f"\nTraining done in {elapsed/60:.1f} min. Final Gaussians: {model.num_gaussians:,}")

# Save log
with open('/content/training_log.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['iteration','loss','psnr','ssim','num_gaussians'])
    w.writerows(log_rows)

In [ ]:
#@title 6. Plot Training Curves
import matplotlib.pyplot as plt
import csv

iters, losses, psnrs, ssims, n_gauss = [], [], [], [], []
with open('/content/training_log.csv') as f:
    for row in csv.DictReader(f):
        iters.append(int(row['iteration']))
        losses.append(float(row['loss']))
        psnrs.append(float(row['psnr']))
        ssims.append(float(row['ssim']))
        n_gauss.append(int(row['num_gaussians']))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(iters, psnrs); axes[0].set_title('PSNR'); axes[0].set_xlabel('Iteration')
axes[1].plot(iters, ssims); axes[1].set_title('SSIM'); axes[1].set_xlabel('Iteration')
axes[2].plot(iters, n_gauss); axes[2].set_title('# Gaussians'); axes[2].set_xlabel('Iteration')
plt.tight_layout(); plt.show()

In [ ]:
#@title 7. Evaluate (PSNR / SSIM / LPIPS)
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
import torchvision.transforms as T
from PIL import Image

EVAL_DIR = '/content/eval_results'
os.makedirs(EVAL_DIR, exist_ok=True)

eval_model = GaussianModel(num_points=1, device=device)
sd = torch.load(f'{CKPT_DIR}/ckpt_final.pt', map_location=device)
eval_model.load_state_dict(sd)
print(f"Loaded {eval_model.num_gaussians:,} Gaussians")

psnr_fn = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim_fn = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
lpips_fn = LearnedPerceptualImagePatchSimilarity(net_type='vgg').to(device)

eval_loader = DataLoader(dataset, batch_size=1, shuffle=False)
save_idx = set(np.linspace(0, len(dataset)-1, 10, dtype=int).tolist())
to_pil = T.ToPILImage()

psnrs, ssims, lpipss = [], [], []
with torch.no_grad():
    a_s = torch.exp(eval_model.scales)
    a_o = torch.sigmoid(eval_model.opacities).squeeze(-1)
    a_q = eval_model.quats / (eval_model.quats.norm(dim=1, keepdim=True)+1e-8)
    means = eval_model.means
    sh = eval_model.sh_coeffs()

    for i, batch in enumerate(tqdm(eval_loader, desc='Evaluating')):
        gt = batch['image'].to(device)
        c2w = batch['c2w'].to(device)
        intr = batch['intrinsics'].to(device)
        H, W = batch['height'].item(), batch['width'].item()
        fid = batch['frame_id'].item()

        vm = torch.inverse(c2w[0])
        fx,fy,cx,cy = intr[0]
        K = torch.zeros((3,3), device=device)
        K[0,0]=fx; K[0,2]=cx; K[1,1]=fy; K[1,2]=cy; K[2,2]=1.0
        bg = torch.tensor(RENDERER_BG_RGB, device=device, dtype=torch.float32)

        colors, _, _ = rasterization(
            means, a_q, a_s, a_o, sh,
            vm.unsqueeze(0), K.unsqueeze(0),
            W, H,
            near_plane=0.01, far_plane=1e10, radius_clip=0.0,
            eps2d=RASTER_EPS2D, sh_degree=SH_DEGREE, backgrounds=bg)
        rendered = colors.permute(0,3,1,2)

        psnrs.append(psnr_fn(rendered, gt).item())
        ssims.append(ssim_fn(rendered, gt).item())
        lpipss.append(lpips_fn(rendered, gt).item())

        if i in save_idx:
            gt_pil = to_pil(gt[0].cpu())
            rn_pil = to_pil(rendered[0].cpu().clamp(0,1))
            comb = Image.new('RGB', (W*2, H))
            comb.paste(gt_pil, (0,0)); comb.paste(rn_pil, (W,0))
            comb.save(f'{EVAL_DIR}/eval_{fid:04d}.png')

print(f"\n{'='*40}")
print(f"Mean PSNR:  {np.mean(psnrs):.2f} dB")
print(f"Mean SSIM:  {np.mean(ssims):.4f}")
print(f"Mean LPIPS: {np.mean(lpipss):.4f}")
print(f"{'='*40}")

In [ ]:
#@title 8. Export .PLY for 3D Viewers
import struct

def export_ply(ckpt_path, out_path):
    sd = torch.load(ckpt_path, map_location='cpu')
    means = sd['means'].float()
    scales = sd['scales'].float()
    quats = sd['quats'].float()
    opacities = sd['opacities'].float()
    if 'sh_dc' in sd:
        sh_coeffs = torch.cat([sd['sh_dc'].float(), sd['sh_rest'].float()], dim=1)
    else:
        sh_coeffs = sd['sh_coeffs'].float()

    N = means.shape[0]
    K = sh_coeffs.shape[1]
    quats = quats / (quats.norm(dim=1, keepdim=True) + 1e-12)

    f_dc = sh_coeffs[:, 0, :]
    f_rest = sh_coeffs[:, 1:, :].reshape(N, -1)
    normals = np.zeros((N, 3), dtype=np.float32)

    data = np.concatenate([
        means.numpy(), normals, f_dc.numpy(), f_rest.numpy(),
        opacities.squeeze(-1).unsqueeze(-1).numpy(),
        scales.numpy(), quats.numpy()
    ], axis=1).astype(np.float32)

    num_rest = f_rest.shape[1]
    header = ['ply', 'format binary_little_endian 1.0', f'element vertex {N}',
              'property float x','property float y','property float z',
              'property float nx','property float ny','property float nz',
              'property float f_dc_0','property float f_dc_1','property float f_dc_2']
    header += [f'property float f_rest_{i}' for i in range(num_rest)]
    header += ['property float opacity',
               'property float scale_0','property float scale_1','property float scale_2',
               'property float rot_0','property float rot_1','property float rot_2','property float rot_3',
               'end_header']

    with open(out_path, 'wb') as f:
        f.write(('\n'.join(header)+'\n').encode('ascii'))
        f.write(data.tobytes())

    print(f"Exported {N:,} Gaussians (SH degree {int(np.sqrt(K))-1}) to {out_path}")
    print(f"  File size: {os.path.getsize(out_path)/1e6:.1f} MB")

export_ply(f'{CKPT_DIR}/ckpt_final.pt', '/content/gaussians.ply')

In [ ]:
#@title 9. Save Results to Google Drive
import shutil

DRIVE_OUT = '/content/drive/MyDrive/SynthSplat_Results'
os.makedirs(DRIVE_OUT, exist_ok=True)

shutil.copy('/content/gaussians.ply', DRIVE_OUT)
print("Copied gaussians.ply")

shutil.copy(f'{CKPT_DIR}/ckpt_final.pt', DRIVE_OUT)
print("Copied ckpt_final.pt")

shutil.copy('/content/training_log.csv', DRIVE_OUT)
print("Copied training_log.csv")

shutil.copytree(EVAL_DIR, f'{DRIVE_OUT}/eval_results', dirs_exist_ok=True)
print("Copied eval_results/")

print(f"\nAll results saved to: Google Drive > SynthSplat_Results")
print(f"  - gaussians.ply  (upload to a 3D viewer)")
print(f"  - ckpt_final.pt  (checkpoint to resume training)")
print(f"  - training_log.csv")
print(f"  - eval_results/  (GT vs rendered comparisons)")